<em style="text-align:center">Copyright Iván Pinar Domínguez</em>

## Importar librerías iniciales e instancia de modelo de chat

In [1]:
from langchain.prompts import PromptTemplate, SystemMessagePromptTemplate,ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import SimpleSequentialChain, LLMChain,TransformChain
f = open(r'C:\Users\joseantonio.clemente\Documents\Langchain\api_key.txt')
api_key = f.read()
llm = ChatOpenAI(openai_api_key=api_key)

## Importamos documentos

In [2]:
from langchain.document_loaders import WikipediaLoader

In [3]:
consulta_wikipedia = input()

In [4]:
idioma_final = input()

In [5]:
loader = WikipediaLoader(query=consulta_wikipedia,lang="es",load_max_docs=10)

In [6]:
data = loader.load()

c:\Users\joseantonio.clemente\AppData\Local\anaconda3\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\joseantonio.clemente\AppData\Local\anaconda3\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


In [7]:
data[0].page_content

'El Club Atlético Osasuna es un club de fútbol de la ciudad de Pamplona, Navarra, que compite en LaLiga EA Sports, la máxima categoría de fútbol en España. Fue fundado el 24 de octubre de 1920, fruto de la fusión de dos clubes: la Sportiva Foot-ball Club y el New Club, por lo que es el club decano de Navarra.[4]\u200b Investigaciones del Archivo Real y General de Navarra indican que el club Sportiva Foot-Ball Club, fundado el 31 de mayo de 1919, cambió de nombre a Club Osasuna el 24 de octubre de 1920; es esta última fecha la que se ha tomado como referencia del nacimiento del club.[5]\u200b En 1926 el nombre de la entidad sufrió una última modificación por la de Club Atlético Osasuna, que se mantiene desde entonces. \nEl Club Atlético Osasuna ha participado en una fase previa de la UEFA Champions League, en cuatro ediciones de la antigua Copa de la UEFA, ahora denominada UEFA Europa League, en la que llegó a las semifinales en la temporada 2006/2007; y en una fase previa de la UEFA Co

In [8]:
texto_entrada = data[0].page_content

# TransformChain

### Definir la función de transformación personalizada

In [9]:
def transformer_function(inputs: dict) -> dict: #Toma de entrada un diccionario y lo devuelve con la transformación oportuna
    texto = inputs['texto']
    primer_parrafo = texto.split('\n')[0]
    return {'salida':primer_parrafo}

In [10]:
transform_chain = TransformChain(input_variables=['texto'],
                                 output_variables=['salida'],
                                 transform=transformer_function)

## Definir la cadena secuencial

In [11]:
#Creamos bloque LLMChain para resumir
template1 = "Crea un resumen en una línea del siguiente texto:\n{texto}"
prompt = ChatPromptTemplate.from_template(template1)
summary_chain = LLMChain(llm=llm,
                     prompt=prompt,
                     output_key="texto_resumen")

C:\Users\joseantonio.clemente\AppData\Local\Temp\ipykernel_11272\2019573838.py:4: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  summary_chain = LLMChain(llm=llm,


In [12]:
#Creamos bloque LLMChain para traducir
template2 = "Traduce a"+ idioma_final + "el siguiente texto:\n{texto}"
prompt = ChatPromptTemplate.from_template(template2)
#prompt.format_prompt(idioma=idioma_final)
translate_chain = LLMChain(llm=llm,
                     prompt=prompt,
                     output_key="texto_traducido")

In [13]:
sequential_chain = SimpleSequentialChain(chains=[transform_chain,summary_chain,translate_chain],
                                        verbose=True)

In [14]:
result = sequential_chain(texto_entrada)

C:\Users\joseantonio.clemente\AppData\Local\Temp\ipykernel_11272\2789050324.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = sequential_chain(texto_entrada)




> Entering new SimpleSequentialChain chain...
El Club Atlético Osasuna es un club de fútbol de la ciudad de Pamplona, Navarra, que compite en LaLiga EA Sports, la máxima categoría de fútbol en España. Fue fundado el 24 de octubre de 1920, fruto de la fusión de dos clubes: la Sportiva Foot-ball Club y el New Club, por lo que es el club decano de Navarra.[4]​ Investigaciones del Archivo Real y General de Navarra indican que el club Sportiva Foot-Ball Club, fundado el 31 de mayo de 1919, cambió de nombre a Club Osasuna el 24 de octubre de 1920; es esta última fecha la que se ha tomado como referencia del nacimiento del club.[5]​ En 1926 el nombre de la entidad sufrió una última modificación por la de Club Atlético Osasuna, que se mantiene desde entonces. 
El Club Atlético Osasuna es el club decano de Navarra, fundado en 1920 y compitiendo en LaLiga EA Sports.
Der Club Atlético Osasuna ist der älteste Verein in Navarra, gegründet im Jahr 1920 und spielt in der LaLiga EA Sports.

> Finish